<a href="https://colab.research.google.com/github/jorh260/gstack/blob/main/churn_proj_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Churn project**

In [ ]:
# starting the project

import pandas as pd
import numpy as np #THE necessary library needed
import joblib
import warnings
warnings.filterwarnings('always')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, average_precision_score
)
from xgboost import XGBClassifier


# ── helper: section banner ──────────────────────────────────────────────────
def print_section(tag, title):
    print(f"\n{'='*65}")
    print(f"  {tag}  |  {title}")
    print(f"{'='*65}\n")


# LOADING DATA ───────────────────────────────────────────────────────────────

raw_data = pd.read_csv(
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d"
    "/master/data/Telco-Customer-Churn.csv"
)

# MESSY DATA CHECK ───────────────────────────────────────────────────────────

def validate_data(df):
    print("=== Missing Values ===")
    print(df.isna().sum())

    print("\n=== Duplicates ===")
    print(df.duplicated().sum())

    print("\n=== Unexpected Categories ===")
    for col in df.select_dtypes(include='object').columns:
        print(f"{col}: {df[col].unique()}")

    print("=== Numeric Conversion Issues ===")
    for col in df.columns:
        converted = pd.to_numeric(df[col], errors='coerce')
        invalids = converted.isna().sum()
        if invalids > 0:
            print(f"{col} {invalids}")


validate_data(raw_data)


# CLEANING DATA ──────────────────────────────────────────────────────────────
# the only data that needs cleaning is TotalCharges.
df = raw_data.copy()

# Convert to numeric, turning blanks/whitespace into NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('\nchecking and extracting the missing values in TotalCharges')
missing_before = df.isnull().sum()
missing_before = missing_before[missing_before > 0]
print(f'missing before:\n{missing_before}')

print('\nhandling those missing values')
for col, cnt in missing_before.items():
    pct = cnt / len(df) * 100
    print(f'   {col:<15s}  {cnt} ({pct:.2f}%) in a row')

# Quick check
print(f"the converted TotalCharges NaN count: {df['TotalCharges'].isna().sum()}")  # should now be 0


# FEATURE SELECTION ──────────────────────────────────────────────────────────
df = df.dropna(subset=['customerID'])

# customerID is just an ID, not a useful feature for prediction
df = df.drop(columns=['customerID'])
print(df.columns)


# PIPELINE ───────────────────────────────────────────────────────────────────
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

x = df.drop(['Churn'], axis=1)
y = df['Churn']

# --- handling missing values in TotalCharges (impute + flag) ---
print('\nFilling the missing values with the median in TotalCharges')

x['TotalCharges_was_missing'] = x['TotalCharges'].isnull().astype(int)

# Note: we don't impute TotalCharges manually here —
# the ColumnTransformer's numerical_transformer (SimpleImputer)
# will handle that automatically inside the pipeline.


# train test split ───────────────────────────────────────────────────────────
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=0,
    stratify=y
)


# categorise the data
cat_cols = [c for c in x.columns if x[c].dtype == 'object']  # used in extracting important features
num_cols = [c for c in x.columns if x[c].dtype != 'object']

# column transformers
numerical_transformer = SimpleImputer(strategy='median')       # fill numeric gaps with the median

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),      # fill text gaps with most common value
    ("onehot",  OneHotEncoder(handle_unknown="ignore")),       # text -> numbers, unseen value -> all zeros
])

# preprocessor
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols),                # "cat" name used later for feature names
], remainder='drop')                                           # drop any columns not in num_cols or cat_cols


# CROSS VALIDATION ───────────────────────────────────────────────────────────
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

#Baseline Model Creation
print("  Running 5-fold cross-validation on BASELINE (Random Forest)...")
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',   # handles class imbalance for RF
        random_state=42,
        n_jobs=-1
    ))
])

rf_cv_scores = cross_val_score(
    baseline_pipeline,
    x, y,
    cv=cv_strategy,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"\n  Random Forest — 5-fold AUC scores:")
for i, score in enumerate(rf_cv_scores, 1):
    bar = '█' * int(score * 50)
    print(f"    Fold {i}: {score:.4f}  {bar}")
print(f"\n  RF Mean AUC : {rf_cv_scores.mean():.4f}")
print(f"  RF Std Dev  : {rf_cv_scores.std():.4f}  (lower = more consistent)")


# BUILDING XGBOOST MODEL with early stopping ─────────────────────────────────

# telling the model to focus more on the churners (the minority in the imbalanced dataset)
# using scale_pos_weight = neg_count / pos_count
neg_weight = (y_train == 0).sum()
pos_weight = (y_train == 1).sum()
scale_pos_weight = neg_weight / pos_weight

print(f"\nimbalance ratio (neg/pos): neg={neg_weight} / pos={pos_weight} = {scale_pos_weight:.2f}")


print('\nBuilding the XGBoost model...')
xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    early_stopping_rounds=20,   # lives here in the constructor, not in .fit()
    random_state=0,
    n_jobs=-1,
    verbosity=0
)


print('\nTransforming train/test sets for XGBoost (manual — needed for early stopping)...')
preprocessor_fitted = preprocessor.fit(x_train)
x_train_transf = preprocessor_fitted.transform(x_train)
x_test_transf  = preprocessor_fitted.transform(x_test)
print(f'  x_train shape after preprocessing : {x_train_transf.shape}')
print(f'  number of features after one-hot  : {x_train_transf.shape[1]}')

print('\nTraining XGBoost model with early stopping...')
xgb_model.fit(
    x_train_transf, y_train,
    eval_set=[(x_test_transf, y_test)],   # validate against the held-out TEST set
    verbose=0
)

num_trees  = xgb_model.best_iteration
best_score = xgb_model.best_score

print('\n___Early Stopping Results___')
print(f'  Trees used   : {num_trees} out of 1000')
print(f'  Best AUC     : {best_score:.4f}')
print(f'  Trees saved  : {1000 - num_trees} (would have been unnecessary)')


print('\nGenerating predictions...')
xgb_preds_proba  = xgb_model.predict_proba(x_test_transf)[:, 1]  # churn probability (class 1)
xgb_preds_binary = (xgb_preds_proba >= 0.5).astype(int)           # 0 or 1 at 0.5 cutoff

print('\nROC-AUC Score')
xgb_auc = roc_auc_score(y_test, xgb_preds_proba)
print(f'  XGBoost AUC     : {xgb_auc:.4f}')
print(f'  Random Forest   : {rf_cv_scores.mean():.4f}')
print(f'  Difference      : {xgb_auc - rf_cv_scores.mean():+.4f}  (XGBoost - RF)')

print(f'\nAverage Precision Score (useful for imbalanced data): '
      f'{average_precision_score(y_test, xgb_preds_proba):.4f}')


# EVALUATION AND INTERPRETATION ──────────────────────────────────────────────
print('\n' + '─'*65)
print('  EVALUATION AND INTERPRETATION')
print('─'*65)

print('\n── Classification Report ──')
print(classification_report(y_test, xgb_preds_binary,
                             target_names=['Retained (0)', 'Churned (1)']))

print('\n── Confusion Matrix ──')
cm = confusion_matrix(y_test, xgb_preds_binary)
tn, fp, fn, tp = cm.ravel()   # flatten the 2x2 matrix to four named values

print("""
  How to read this table:
    TN = correctly predicted "will stay"   (good)
    FP = wrongly flagged as "will churn"   (false alarm — costs money)
    FN = missed churners                   (bad — lost customers)
    TP = correctly caught churners         (good — saved revenue)
""")
print(f'  {"":25s}  {"Predicted: Stay":>15s}   {"Predicted: Churn":>16s}')
print(f'  {"─"*62}')
print(f'  {"Actual: Stay (did not churn)":25s}  {tn:^15d}   {fp:^16d}')
print(f'  {"Actual: Churn (left)":25s}  {fn:^15d}   {tp:^16d}')
print(f'  {"─"*62}')
print(f'\n  TN={tn}  FP={fp}  FN={fn}  TP={tp}')


# BUSINESS IMPACT ANALYSIS ───────────────────────────────────────────────────
print('\n' + '─'*65)
print('  BUSINESS IMPACT ANALYSIS')
print('─'*65)

# Business variables
avg_customer_ltv      = 1500   # average lifetime value of a retained customer ($)
retention_offer_cost  = 100    # cost to send a retention offer (discount, upgrade, etc.) ($)
cost_false_alarm      = 50     # cost of contacting a customer who was never going to leave ($)

# Derived calculations
revenue_saved     = tp * avg_customer_ltv              # only TRUE churners generate saved revenue
total_offer_cost  = (tp + fp) * retention_offer_cost   # offers are sent to everyone flagged (TP + FP)
false_alarm_cost  = fp * cost_false_alarm              # extra waste on non-churners specifically
net_value         = revenue_saved - total_offer_cost   # what you save minus what you spend

print(f'\n  {"Metric":<22s}  {"Value":>12s}   Notes')
print(f'  {"─"*65}')
print(f'  {"Revenue Saved":<22s}  ${revenue_saved:>11,.2f}   '
      f'({tp} churners retained × ${avg_customer_ltv:,} LTV)')
print(f'  {"Total Offer Cost":<22s}  ${total_offer_cost:>11,.2f}   '
      f'({tp + fp} offers sent × ${retention_offer_cost} each)')
print(f'  {"False Alarm Cost":<22s}  ${false_alarm_cost:>11,.2f}   '
      f'({fp} non-churners contacted × ${cost_false_alarm} each)')
print(f'  {"─"*65}')
print(f'  {"Net Value":<22s}  ${net_value:>11,.2f}')

print("""
  ── Without the Model ──

  If the retention team contacted EVERY customer:
    → Huge outreach cost (all customers × retention offer cost).

  If the team contacted NO ONE:
    → All churners leave silently, losing their full lifetime value.

  In both cases, the business either overspends or loses revenue.

  ── Threshold Note ──
  We used 0.5 as the churn probability cutoff.
  Adjusting this threshold changes the trade-off:
    · Raise threshold → fewer false alarms, lower costs.
    · Lower threshold → catch more churners, higher saved revenue.
  The optimal threshold depends on business priorities and budget.
""")


# FEATURE IMPORTANCE ─────────────────────────────────────────────────────────
print('\n' + '─'*65)
print('  TOP FEATURES DRIVING CHURN')
print('─'*65)

# Get feature names after one-hot encoding
ohe_feature_names = (
    preprocessor_fitted
    .named_transformers_['cat']      # matches ColumnTransformer name "cat"
    .named_steps['onehot']           # matches Pipeline step name "onehot"
    .get_feature_names_out(cat_cols)
)

all_feature_names = num_cols + list(ohe_feature_names)

importances = pd.Series(
    xgb_model.feature_importances_,
    index=all_feature_names
).sort_values(ascending=False)

print(f"\n  {'Rank':<6s} {'Feature':<45s} {'Importance':>10s}   Bar")
print(f"  {'─'*80}")
for rank, (feat, imp) in enumerate(importances.head(15).items(), 1):
    bar = '▮' * int(imp * 300)
    print(f"  {rank:<6d} {feat:<45s} {imp:>10.4f}   {bar}")

print("""
  ── Business Interpretation ──

  Monthly Charges       → High bills strongly linked to churn.
                          Action: Offer flexible pricing plans.

  Tenure                → New customers churn more.
                          Action: Early loyalty rewards and onboarding programs.

  Contract Type         → Month-to-month customers churn more.
                          Action: Incentivize longer contracts with discounts.

  Internet Service      → Fiber optic users show higher churn risk.
                          Action: Prioritize service quality and support.

  Payment Method        → Electronic check users churn more.
                          Action: Promote auto-pay / credit card options.

  Senior Citizen        → Distinct churn pattern in older customers.
                          Action: Simplified plans and tailored communication.

  Tech Support          → Lack of support correlates with churn.
                          Action: Upsell support and security bundles.
""")


# PRODUCTION INFERENCE — PREDICTING NEW CUSTOMERS ────────────────────────────
print_section("SECTION 11", "Production Inference — Predicting New Customers")

print("""
  In production, this model runs nightly on the full customer database.
  The system produces a churn probability for each customer, ranks them,
  and the top-N are sent to the retention team's CRM queue each morning.

  Here's what a single inference looks like:
""")

# Profile 1: High-risk customer
high_risk = pd.DataFrame([{
    'gender': 'Female', 'SeniorCitizen': 0, 'Partner': 'No',
    'Dependents': 'No', 'tenure': 1,                          # brand new
    'PhoneService': 'Yes', 'MultipleLines': 'No',
    'InternetService': 'Fiber optic',                         # expensive service
    'OnlineSecurity': 'No', 'OnlineBackup': 'No',
    'DeviceProtection': 'No', 'TechSupport': 'No',
    'StreamingTV': 'No', 'StreamingMovies': 'No',
    'Contract': 'Month-to-month',                             # no lock-in
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 94.95,
    'TotalCharges': 94.95,                                    # just one month
    'TotalCharges_was_missing': 0                             # TotalCharges is present → flag = 0
}])

# Profile 2: Low-risk customer
low_risk = pd.DataFrame([{
    'gender': 'Male', 'SeniorCitizen': 0, 'Partner': 'Yes',
    'Dependents': 'Yes', 'tenure': 48,                       # 4 years
    'PhoneService': 'Yes', 'MultipleLines': 'Yes',
    'InternetService': 'DSL',
    'OnlineSecurity': 'Yes', 'OnlineBackup': 'Yes',
    'DeviceProtection': 'Yes', 'TechSupport': 'Yes',
    'StreamingTV': 'Yes', 'StreamingMovies': 'Yes',
    'Contract': 'Two year',                                   # locked in
    'PaperlessBilling': 'No',
    'PaymentMethod': 'Bank transfer (automatic)',
    'MonthlyCharges': 80.10,
    'TotalCharges': 3844.8,
    'TotalCharges_was_missing': 0                             # TotalCharges is present → flag = 0
}])

for label, profile in [('HIGH-RISK CUSTOMER PROFILE', high_risk),
                        ('LOW-RISK CUSTOMER PROFILE',  low_risk)]:
    proc = preprocessor_fitted.transform(profile)
    prob = xgb_model.predict_proba(proc)[0][1]
    risk = 'HIGH   🔴' if prob > 0.6 else ('MEDIUM 🟡' if prob > 0.3 else 'LOW    🟢')

    action = (
        "PRIORITY CONTACT: Offer 20% loyalty discount + annual contract upgrade"
        if prob > 0.6 else
        "MONITOR: Include in monthly check-in campaign"
        if prob > 0.3 else
        "NO ACTION: Stable customer — standard comms only"
    )

    print(f"  {'─'*60}")
    print(f"  {label}")
    print(f"  {'─'*60}")
    print(f"  Tenure          : {profile['tenure'].values[0]} month(s)")
    print(f"  Contract        : {profile['Contract'].values[0]}")
    print(f"  Internet        : {profile['InternetService'].values[0]}")
    print(f"  Churn prob.     : {prob:.1%}")
    print(f"  Risk level      : {risk}")
    print(f"  Recommended     : → {action}")
    print()

# ── End of project ──────────────────────────────────────────────────────────
# Next task: explain each section and sub-section in depth
# to build an intuitive mental picture of both the ML workflow
# and the business impact layer — so the full process can be
# visualised mentally without needing to re-read the code.

=== Missing Values ===
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

=== Duplicates ===
0

=== Unexpected Categories ===
customerID: ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
gender: ['Female' 'Male']
Partner: ['Yes' 'No']
Dependents: ['No' 'Yes']
PhoneService: ['No' 'Yes']
MultipleLines: ['No phone service' 'No' 'Yes']
InternetService: ['DSL' 'Fiber optic' 'No']
OnlineSecurity: ['No' 'Yes' 'No internet service']
OnlineBackup: ['Yes' 'No' 'No internet service']
DeviceProtection: ['No' 'Yes' 'No internet service']
TechSupp